In [19]:
import pyarrow as pa
pa.__version__

'24.0.0'

## PyArrow RecordBatch Memory Layout


In [20]:
table = pa.table({
    "id": [1, 2, 3, 4],
    "Name": ["Robert", "Alberto", "Piotr", "Marco"],
    "Country": ["UK", "Portugal", "Poland", "Italy"],
    "Cost": [550, 540, 200, 690],
})
table

pyarrow.Table
id: int64
Name: string
Country: string
Cost: int64
----
id: [[1,2,3,4]]
Name: [["Robert","Alberto","Piotr","Marco"]]
Country: [["UK","Portugal","Poland","Italy"]]
Cost: [[550,540,200,690]]

PyArrow Table can hold multiple RecordBatches.
Our simple example table holds only batch of data and that is what we are going to use.

In [21]:
table.to_batches()

[pyarrow.RecordBatch
 id: int64
 Name: string
 Country: string
 Cost: int64
 ----
 id: [1,2,3,4]
 Name: ["Robert","Alberto","Piotr","Marco"]
 Country: ["UK","Portugal","Poland","Italy"]
 Cost: [550,540,200,690]]

In [22]:
batch = table.to_batches()[0]
batch

pyarrow.RecordBatch
id: int64
Name: string
Country: string
Cost: int64
----
id: [1,2,3,4]
Name: ["Robert","Alberto","Piotr","Marco"]
Country: ["UK","Portugal","Poland","Italy"]
Cost: [550,540,200,690]

## PyArrow buffers

We can inspect Arrow layout via the number of allocated buffers.

In [23]:
int_arr = batch[0]
int_arr

[
  1,
  2,
  3,
  4
]

Integer array without missing values only has one buffer with actual values allocated. The validity bitmap is None, meaning it is not allocated.

In [24]:
int_arr.buffers()

[None,
 <pyarrow.Buffer address=0x300040340 size=32 is_cpu=True is_mutable=True>]

In [25]:
str_arr = batch[2]
str_arr

[
  "UK",
  "Portugal",
  "Poland",
  "Italy"
]

String array holds more buffers:
- first is the validity bitmap which is not allocated,
- second are the values
- and the third are the offsets that explain where each element starts end ends.

In [26]:
str_arr.buffers()

[None,
 <pyarrow.Buffer address=0x300040440 size=20 is_cpu=True is_mutable=True>,
 <pyarrow.Buffer address=0x3000404c0 size=21 is_cpu=True is_mutable=True>]

## Nanoarrow

Nanoarrow makes it much easier to inspect the layout as it actually prints out the values of the buffers.

In [27]:
import nanoarrow as na

In [28]:
na.array(int_arr).inspect()

<ArrowArray int64>
- length: 4
- offset: 0
- null_count: 0
- buffers[2]:
  - validity <bool[0 b] >
  - data <int64[32 b] 1 2 3 4>
- dictionary: NULL
- children[0]:


In [29]:
na.array(str_arr).inspect()

<ArrowArray string>
- length: 4
- offset: 0
- null_count: 0
- buffers[3]:
  - validity <bool[0 b] >
  - data_offset <int32[20 b] 0 2 10 16 21>
  - data <string[21 b] b'UKPortugalPolandItaly'>
- dictionary: NULL
- children[0]:


In [30]:
str_1_arr = batch[1]
na.array(str_1_arr).inspect()

<ArrowArray string>
- length: 4
- offset: 0
- null_count: 0
- buffers[3]:
  - validity <bool[0 b] >
  - data_offset <int32[20 b] 0 6 13 18 23>
  - data <string[23 b] b'RobertAlbertoPiotrMarco'>
- dictionary: NULL
- children[0]:
